# MovieLens 25M full training run

Run the cells from top to bottom in a Google Colab runtime. Colab uses `/content` paths; place the six CSV files in `MyDrive/recsys/data/raw/`. This workflow loads the full dataset into SQLite, trains all classical models, and keeps MLflow and exported results on Drive.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/sxxxsewan/MovieRecSys.git"
REPO_DIR = Path("/content/MovieRecSys")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive" / "recsys"
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "raw"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"
MLFLOW_DIR = DRIVE_PROJECT_DIR / "mlruns"
DATABASE_PATH = DATA_DIR / "recsys.sqlite"
REGISTRY_PATH = DRIVE_PROJECT_DIR / "mlruns_registry.db"
CSV_FILES = (
    "ratings.csv",
    "movies.csv",
    "tags.csv",
    "links.csv",
    "genome-scores.csv",
    "genome-tags.csv",
)
print(f"Repository: {REPO_DIR}")
print(f"Drive CSV source: {DRIVE_DATA_DIR}")
print(f"SQLite database: {DATABASE_PATH}")
print(f"Drive outputs: {OUTPUT_DIR}")

Repository: /content/MovieRecSys
Drive CSV source: /content/drive/MyDrive/recsys/data/raw
SQLite database: /content/MovieRecSys/data/recsys.sqlite
Drive outputs: /content/drive/MyDrive/recsys/outputs


## 1. Setup

Clone or update the repository and install its pinned project and development dependencies.

In [2]:
import subprocess
import sys

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    cwd=REPO_DIR,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.[dev]'], returncode=0)

## 2. Data

Mount Drive, validate every required MovieLens CSV, and copy the files to the repository's configured data directory.

In [3]:
import shutil

from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
missing_files = [name for name in CSV_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing MovieLens CSV files in {DRIVE_DATA_DIR}: {', '.join(missing_files)}. "
        "Upload the six files to MyDrive/recsys/data/raw/ and rerun this cell."
    )
DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in CSV_FILES:
    shutil.copy2(DRIVE_DATA_DIR / filename, DATA_DIR / filename)
print(f"Copied {len(CSV_FILES)} CSV files into {DATA_DIR}")

Mounted at /content/drive
Copied 6 CSV files into /content/MovieRecSys/data


In [4]:
!ls -la {DATA_DIR}
!cd {REPO_DIR} && {sys.executable} -c "import recsys; print('recsys OK')"

total 1129596
drwxr-xr-x  2 root root      4096 Oct 28 12:30 .
drwxr-xr-x 11 root root      4096 Oct 28 12:28 ..
-rw-------  1 root root 435164157 Sep 13  2021 genome-scores.csv
-rw-------  1 root root     18103 Sep 13  2021 genome-tags.csv
-rw-------  1 root root   1368578 Sep 13  2021 links.csv
-rw-------  1 root root   3038099 Sep 13  2021 movies.csv
-rw-------  1 root root 678260987 Sep 13  2021 ratings.csv
-rw-r--r--  1 root root     10460 Oct 28 12:28 README.txt
-rw-------  1 root root  38810332 Sep 13  2021 tags.csv
recsys OK


In [5]:
"""result = subprocess.run(
    [
        sys.executable,
        "-m",
        "recsys.data.loader",
        "--config",
        "configs/data.yaml",
        "--replace",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print("return code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-3000:]) """

'result = subprocess.run(\n    [sys.executable, "-m", "recsys.data.loader", "--config", "configs/data.yaml", "--replace"],\n    cwd=REPO_DIR,\n    capture_output=True,\n    text=True,\n)\nprint("return code:", result.returncode)\nprint(result.stdout[-3000:])\nprint(result.stderr[-3000:]) '

## 3. Configuration and full run

Use SQLite for the application database and Drive-backed file stores for MLflow tracking, registry, and model artifacts. Training now fits and releases models sequentially, shares the compact training-history mapping, limits neighbor-search working memory and BLAS threads, and batches evaluation to reduce peak RAM. Apply migrations, load the full CSV dataset, then train and evaluate all models.

In [6]:
import os
import subprocess
import sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MOVIELENS_DATA_DIR"] = str(DATA_DIR)
os.environ["DATABASE_URL"] = f"sqlite:///{DATABASE_PATH}"
os.environ["MLFLOW_TRACKING_URI"] = f"file:{MLFLOW_DIR}"
os.environ["MLFLOW_REGISTRY_URI"] = f"sqlite:///{REGISTRY_PATH}"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["BLIS_NUM_THREADS"] = "1"

subprocess.run(["alembic", "upgrade", "head"], cwd=REPO_DIR, check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "recsys.data.loader",
        "--config",
        "configs/data.yaml",
        "--replace",
    ],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "recsys.training",
        "--data-config",
        "configs/data.yaml",
        "--model-config",
        "configs/models.yaml",
        "--experiment-config",
        "configs/experiments.yaml",
    ],
    cwd=REPO_DIR,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'recsys.training', '--data-config', 'configs/data.yaml', '--model-config', 'configs/models.yaml', '--experiment-config', 'configs/experiments.yaml'], returncode=0)

## 4. Results and saved artifacts

Show the latest MLflow run for each model, save validation and test metrics to Drive, export the validation-selected model, and save an interactive comparison plot.

In [7]:
import mlflow
import pandas as pd
import plotly.express as px
from mlflow import MlflowClient

experiment_name = "movielens-25m"
client = MlflowClient(tracking_uri=os.environ["MLFLOW_TRACKING_URI"])
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise RuntimeError(f"MLflow experiment not found: {experiment_name}")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=100,
)
latest_runs = {}
for run in runs:
    latest_runs.setdefault(run.info.run_name or run.info.run_id, run)
metrics_frame = pd.DataFrame(
    [
        {"model": name, "run_id": run.info.run_id, **run.data.metrics}
        for name, run in latest_runs.items()
    ]
)
if metrics_frame.empty:
    raise RuntimeError("No completed model runs were found in MLflow")
metrics_frame = metrics_frame.sort_values(
    "validation_ndcg_at_10", ascending=False
).reset_index(drop=True)
metrics_path = OUTPUT_DIR / "model_metrics.csv"
metrics_frame.to_csv(metrics_path, index=False)
display(metrics_frame)

best_run = max(
    latest_runs.values(),
    key=lambda run: run.data.metrics.get("validation_ndcg_at_10", float("-inf")),
)
best_model_dir = OUTPUT_DIR / "best_model"
downloaded_model = mlflow.artifacts.download_artifacts(
    run_id=best_run.info.run_id,
    artifact_path="model",
    dst_path=str(best_model_dir),
)
plot_path = OUTPUT_DIR / "model_metrics.html"
figure = px.bar(
    metrics_frame,
    x="model",
    y="test_ndcg_at_10",
    color="validation_ndcg_at_10",
    title="MovieLens 25M model comparison: NDCG@10",
)
figure.write_html(plot_path)
display(figure)
print(f"Metrics CSV: {metrics_path}")
print(f"Interactive plot: {plot_path}")
print(f"Best model ({best_run.info.run_name}) exported to: {downloaded_model}")
print(f"MLflow run artifacts remain under: {MLFLOW_DIR}")

,model,run_id,validation_ndcg_at_10,validation_recall_at_10,validation_map_at_10,validation_precision_at_10,validation_coverage,validation_novelty,validation_average_popularity,test_ndcg_at_10,test_recall_at_10,test_map_at_10,test_precision_at_10,test_coverage,test_novelty,test_average_popularity
0,svds_reference,c4532e8404ab469facbbe8f370f63fc3,0.138089,0.058598,0.075312,0.118621,0.017221,10.029561,23701.507393,0.105820,0.056776,0.055840,0.086601,0.016693,10.045339,23468.711170
1,hybrid,a0919c37275442889957d7dc4806498a,0.120641,0.052640,0.060294,0.101987,0.036958,12.766150,9802.605465,0.090834,0.056750,0.044828,0.073733,0.034346,12.789468,9685.205731
2,item_knn,95d68c1b7c4f410cb2467818b5df5b28,0.106026,0.042344,0.055502,0.093680,0.010493,9.685376,28321.386884,0.084454,0.042683,0.043951,0.069859,0.010060,9.680548,28374.510958
3,popularity,fb1ad05b881448cdab5666a61b010a08,0.075539,0.027733,0.038191,0.062937,0.004486,8.779198,46940.359996,0.062129,0.030792,0.031145,0.050199,0.004389,8.800857,46356.637172
4,content_based,b833d9d593464fa8be493a83d60c7ed1,0.073192,0.035709,0.030897,0.070012,0.034042,16.643563,6830.312579,0.053499,0.037242,0.022801,0.049005,0.031591,16.694812,6707.831786
5,als,1fae7c77458f49469fa81480d6fde5ee,0.000381,0.000097,0.000125,0.000358,0.044086,18.972745,41.828776,0.000403,0.000131,0.000118,0.000398,0.041043,18.992687,41.398780


Metrics CSV: /content/drive/MyDrive/recsys/outputs/model_metrics.csv
Interactive plot: /content/drive/MyDrive/recsys/outputs/model_metrics.html
Best model (svds_reference) exported to: /content/drive/MyDrive/recsys/outputs/best_model/model
MLflow run artifacts remain under: /content/drive/MyDrive/recsys/mlruns
